# Baseline Retrieval Models

We implement two classical (non-neural) baselines to compare against our trained bi-encoder:

| Baseline | Method |
|----------|--------|
| **TF-IDF** | Sparse vector space model; scores documents by term frequency weighted by inverse document frequency |
| **BM25** | Probabilistic extension of TF-IDF with document-length normalisation; industry standard for keyword search |

**Evaluation metrics:**
- **Recall@k** — fraction of queries where the correct passage appears in the top-k results
- **MRR** (Mean Reciprocal Rank) — average of 1/rank of the first correct result

Both metrics are computed on the held-out **test split** from the data-preparation step.

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

PROCESSED_DIR = Path('../data/processed')
EVAL_DIR      = Path('../evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Load corpus
with open(PROCESSED_DIR / 'corpus.json', encoding='utf-8') as f:
    corpus = json.load(f)   # {doc_id: text}

# Load test split
test_triplets = []
with open(PROCESSED_DIR / 'test.jsonl', encoding='utf-8') as f:
    for line in f:
        test_triplets.append(json.loads(line))

# Build ordered lists for indexing
doc_ids   = list(corpus.keys())
doc_texts = [corpus[did] for did in doc_ids]
id_to_idx = {did: i for i, did in enumerate(doc_ids)}

print(f'Corpus size:  {len(corpus):,} passages')
print(f'Test queries: {len(test_triplets):,}')

## 2. Evaluation Helper

In [ ]:
def evaluate(model_name: str, retrieve_fn, triplets: list, ks=(1, 5, 10)) -> dict:
    """
    retrieve_fn(query: str) -> list of doc_ids ordered by score (best first)
    Returns dict with Recall@k and MRR.
    """
    reciprocal_ranks = []
    hits = {k: 0 for k in ks}

    for t in tqdm(triplets, desc=f'Evaluating {model_name}'):
        query      = t['query']
        correct_id = t['positive_id']

        ranked = retrieve_fn(query)

        # Reciprocal rank
        rr = 0.0
        for rank, did in enumerate(ranked[:max(ks)], start=1):
            if did == correct_id:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

        # Recall@k
        for k in ks:
            if correct_id in ranked[:k]:
                hits[k] += 1

    n = len(triplets)
    results = {f'Recall@{k}': hits[k] / n for k in ks}
    results['MRR'] = float(np.mean(reciprocal_ranks))
    return results


all_results = {}   # model_name -> metrics dict
print('Evaluation helper ready.')

## 3. TF-IDF Baseline

We fit a TF-IDF vectorizer on the entire corpus, then at query time:
1. Transform the query with the same vectorizer
2. Compute cosine similarity against all document vectors
3. Return documents ranked by similarity

In [ ]:
print('Fitting TF-IDF on corpus...')
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=50_000,
    sublinear_tf=True,      # log(1 + tf) dampening
    ngram_range=(1, 2),     # unigrams + bigrams
    min_df=2,
)
doc_matrix = tfidf.fit_transform(doc_texts)   # shape: (num_docs, vocab)

print(f'Done in {time.time()-t0:.1f}s | Matrix shape: {doc_matrix.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')

In [ ]:
def tfidf_retrieve(query: str, k: int = 10) -> list:
    q_vec  = tfidf.transform([query])                     # (1, vocab)
    scores = cosine_similarity(q_vec, doc_matrix).ravel() # (num_docs,)
    top_k  = np.argsort(scores)[::-1][:k]
    return [doc_ids[i] for i in top_k]


# Quick sanity check
sample_q = test_triplets[0]['query']
sample_results = tfidf_retrieve(sample_q, k=3)
print(f'Query:   {sample_q}')
print(f'Top-3 doc IDs: {sample_results}')
print(f'Correct doc:   {test_triplets[0]["positive_id"]}')
print(f'Hit: {test_triplets[0]["positive_id"] in sample_results}')

In [ ]:
tfidf_results = evaluate('TF-IDF', tfidf_retrieve, test_triplets)
all_results['TF-IDF'] = tfidf_results

print('\nTF-IDF Results:')
for k, v in tfidf_results.items():
    print(f'  {k}: {v:.4f}')

## 4. BM25 Baseline

BM25 (Best Match 25) is a probabilistic ranking function. Key improvements over TF-IDF:
- **Term frequency saturation**: adding the same term repeatedly has diminishing returns
- **Document length normalisation**: shorter documents are not penalised for fewer term occurrences

We use `BM25Okapi` (the standard Okapi BM25 variant) from the `rank-bm25` library.

In [ ]:
print('Building BM25 index...')
t0 = time.time()

tokenized_corpus = [doc.lower().split() for doc in tqdm(doc_texts, desc='Tokenising')]
bm25 = BM25Okapi(tokenized_corpus)

print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
def bm25_retrieve(query: str, k: int = 10) -> list:
    tokenized_q = query.lower().split()
    scores = bm25.get_scores(tokenized_q)   # (num_docs,)
    top_k  = np.argsort(scores)[::-1][:k]
    return [doc_ids[i] for i in top_k]


# Quick sanity check
sample_results_bm25 = bm25_retrieve(sample_q, k=3)
print(f'Query:   {sample_q}')
print(f'Top-3 doc IDs: {sample_results_bm25}')
print(f'Correct doc:   {test_triplets[0]["positive_id"]}')
print(f'Hit: {test_triplets[0]["positive_id"] in sample_results_bm25}')

In [ ]:
bm25_results = evaluate('BM25', bm25_retrieve, test_triplets)
all_results['BM25'] = bm25_results

print('\nBM25 Results:')
for k, v in bm25_results.items():
    print(f'  {k}: {v:.4f}')

## 5. Results Comparison

In [ ]:
# Print comparison table
metrics = ['Recall@1', 'Recall@5', 'Recall@10', 'MRR']
header  = f"{'Model':<20}" + ''.join(f"{m:<14}" for m in metrics)
print(header)
print('-' * len(header))
for model, res in all_results.items():
    row = f"{model:<20}" + ''.join(f"{res[m]:<14.4f}" for m in metrics)
    print(row)

# Save for the final evaluation comparison
with open(EVAL_DIR / 'baseline_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {EVAL_DIR / "baseline_results.json"}')

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
models  = list(all_results.keys())
colors  = ['steelblue', 'coral']

for ax, metric in zip(axes, metrics):
    vals = [all_results[m][metric] for m in models]
    bars = ax.bar(models, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11)

plt.suptitle('Baseline Retrieval — TF-IDF vs BM25', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 6. Qualitative Examples

In [ ]:
def show_results(query: str, positive_id: str, k: int = 3):
    print(f'QUERY: {query}')
    print(f'CORRECT DOC: {positive_id}')
    print()

    for model_name, retrieve_fn in [('TF-IDF', tfidf_retrieve), ('BM25', bm25_retrieve)]:
        ranked = retrieve_fn(query, k=k)
        print(f'--- {model_name} Top-{k} ---')
        for rank, did in enumerate(ranked, 1):
            hit = '✓' if did == positive_id else ' '
            print(f'  [{hit}] Rank {rank} ({did}): {corpus[did][:120]}...')
        print()


print('=== Example 1 — hit case ===')
# Find a case where BM25 gets it right
for t in test_triplets[:200]:
    if t['positive_id'] in bm25_retrieve(t['query'], k=3):
        show_results(t['query'], t['positive_id'])
        break

print('=== Example 2 — miss case ===')
# Find a case where both miss
for t in test_triplets[:200]:
    bm25_top = bm25_retrieve(t['query'], k=5)
    tfidf_top = tfidf_retrieve(t['query'], k=5)
    if t['positive_id'] not in bm25_top and t['positive_id'] not in tfidf_top:
        show_results(t['query'], t['positive_id'])
        break

## 7. Summary

| Aspect | TF-IDF | BM25 |
|--------|--------|------|
| **Method** | Cosine similarity over TF-IDF vectors | Probabilistic ranking with TF saturation |
| **Strengths** | Fast, simple, interpretable | Better than TF-IDF on most IR benchmarks |
| **Weaknesses** | Sensitive to document length; no semantic understanding | Still purely lexical — misses paraphrases |
| **Vocabulary mismatch** | Fails when query uses different words than passage | Same |

**Why these baselines are not enough:**  
Both methods rely entirely on **exact word overlap**. If the query says *"How does self-attention work?"* and the passage says *"The model focuses on relevant tokens using a weighted sum"*, neither TF-IDF nor BM25 will match them — there are no shared content words.  
This is the core motivation for training a neural bi-encoder.

**Next step → Bi-Encoder model training**